# Current Qian PFC workflow
This notebook loads raw counts, fits the selected shared log-E Stage 1, and derives Fisher dynamics. There is no Stage 1b or independent Stage 2 refit.

Install once in your environment: `pip install -e ".[qian,notebooks]"`. Set the data path and a new run directory below. Run all cells on the cluster containing the Qian data.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if not (ROOT / "src/flow_stp").exists():
    raise RuntimeError("Open this notebook from inside the repository.")
sys.path.insert(0, str(ROOT / "src"))
from flow_stp.data import QianPFCConfig, load_qian_pfc_for_stage1, validate_qian_pfc_data
from flow_stp.stage1 import Stage1ARunConfig, run_stage1a
from flow_stp.stage2 import estimate_local_dynamics, integrate_flow_rk4
import numpy as np
import torch


In [ ]:
DATA_ROOT = Path("/scratch/uchitra1/datasets/spatial-RNA/merfish/merfish_cortex_atlas/zenodo_14422018_processed")
OUTPUT = ROOT / "experiments/qian_selected_logE"
qian = load_qian_pfc_for_stage1(QianPFCConfig(data_root=DATA_ROOT))
validate_qian_pfc_data(qian)
print(qian.num_slices, qian.num_cells, qian.num_genes)


In [ ]:
result = run_stage1a(**qian.as_stage1_kwargs(), config=Stage1ARunConfig(output_dir=OUTPUT))
print(result.summary)
display(result.coordinate_diagnostics)


## Derived dynamics
Times below are normalized to [0,1]. Velocity is in learned z units; interpreting physical motion requires a chart lift. Use the metrics notebook for tissue displacement evaluation.

In [ ]:
device = next(result.model.parameters()).device
z = torch.linspace(0.05, 0.95, 64, device=device)
t = torch.full_like(z, 0.5)
local = estimate_local_dynamics(result.model.fisher_flat, z, t)
trajectory = integrate_flow_rk4(result.model.fisher_flat, z, t0=0.0, t1=1.0)
print("Velocity range:", local.velocity.min().item(), local.velocity.max().item())
print("Final positions:", trajectory.final_position[:5])
